# Nonlinear Multi-Agent Influence Networks
**Nanda & Nagpall** — Interactive exploration notebook

GitHub: https://github.com/mihiit/NonLinear_Multi-Agent_influencenetworks

This notebook walks through the key experiments and theoretical quantities from the paper.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt

from src.dynamics  import simulate, steady_state, transient_growth_rate
from src.networks  import twitter_graph, star, complete, erdos_renyi, spectral_stats
from src.theory    import theory_summary, spectral_threshold, decoupling_gap
from src.experiments import run_topology_sweep, run_real_network_sweep
from src.agent_experiment import run_agent_experiment, early_late_summary
from src.plotting  import plot_topology_sweep, plot_agent_experiment

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Quick theory check — decoupling gap on Twitter network

In [ ]:
A = twitter_graph(n=200, seed=0)
s = theory_summary(A, alpha=0.05, p=0.5, r=0.02)
for k, v in s.items():
    print(f'{k:30s}: {v:.6f}' if isinstance(v, float) else f'{k:30s}: {v}')

## 2. Single simulation trajectory

In [ ]:
A     = erdos_renyi(n=60, p=0.1, seed=0)
alpha = 0.3
X     = simulate(A, alpha=alpha, p=0.5, r=0.02, T=200)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot(X.mean(axis=1), color='steelblue', lw=1.5)
axes[0].set(xlabel='Time $t$', ylabel='Mean activation $\\bar{x}(t)$',
            title=f'alpha={alpha}')
axes[1].imshow(X.T, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
axes[1].set(xlabel='Time', ylabel='Agent', title='Agent belief heatmap')
plt.tight_layout()

## 3. Topology sweep (quick, 10 trials)

In [ ]:
alpha_vals = np.arange(0, 0.8, 0.04)
results = run_topology_sweep(
    alpha_vals, topologies=['star', 'er', 'dense', 'ba'],
    n=40, p=0.5, r=0.02, T=150, n_trials=20, n_graphs=5
)
plot_topology_sweep(results, save_path='../figures/fig_topology_quick.jpg')
plt.show()

## 4. AI agent misinformation experiment

In [ ]:
agent_results = run_agent_experiment(n=30, alpha=0.12, p=0.6, r=0.03, T=120)
summary = early_late_summary(agent_results)

for name, vals in summary.items():
    print(f'{name:10s}  early t=20: {vals["early"]:.3f}   late t=120: {vals["late"]:.3f}')

plot_agent_experiment(agent_results)

## 5. Lyapunov contraction constant
Verify that the explicit $c$ from Theorem 4.1 is positive in the strengthened regime.

In [ ]:
from src.theory import lyapunov_constant, lyapunov_regime_satisfied

A = complete(30)
for alpha in [0.001, 0.005, 0.01, 0.02]:
    c    = lyapunov_constant(A, alpha, p=0.5, r=0.02)
    ok   = lyapunov_regime_satisfied(A, alpha, p=0.5, r=0.02)
    print(f'alpha={alpha:.3f}  regime_ok={ok}  c={c:.4f}')